# `pls_free_knots`

Simple example that:

1. Instantiates the paired power-law + spline mass model with **free (inferred) knot positions**, `pls_free_knots`.
2. Draws `knot_logits` and `spline_coefficients` from simple priors, mimicking an MCMC step.
3. Plots the mass PDF

## Instantiate the model


In [ ]:
import os, sys
os.environ['CHIMERA_ENABLE_GPU'] = "False"
sys.path.append(os.getcwd()+'/../')
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from CHIMERA.mass.paired import pls_free_knots
from CHIMERA.mass.paired.pls_free_knots import knot_logits_to_positions  # reuse for plotting knot positions
from CHIMERA.mass import p_m1  # analytic marginal p(m1), accounts for the pairing function
import seaborn as sns

# This particular draw's mass-function support (must lie within [knots_low, knots_high])
m_low, m_high = 5.1, 150.0

num_inner_knots = 8
degree = 3

mass = pls_free_knots(
    m_low=m_low, m_high=m_high,
    alpha=3.4, beta=1.1, delta_m=4.8,
    min_knot = 5.5, max_knot=80.,
    num_inner_knots=num_inner_knots, degree=degree,
)

print(f"num_coeffs      = {mass.num_coeffs}")
print(f"knot_logits shape = {mass.knot_logits.shape}")
print(f"spline_coefficients shape = {mass.spline_coefficients.shape}")

## Draw `knot_logits` and `spline_coefficients` from priors and plot realziations

In [ ]:
def draw_realization(key, mass, scale_knots, scale_coeffs):
    k1, k2 = jax.random.split(key)
    knot_logits = scale_knots * jax.random.normal(k1, shape=(mass.num_inner_knots - 1,))
    spline_coefficients = scale_coeffs * jax.random.normal(k2, shape=(mass.num_coeffs,))
    mass_i = mass.update(knot_logits=knot_logits, spline_coefficients=spline_coefficients)
    knot_positions = knot_logits_to_positions(
        knot_logits, mass.min_knot, mass.max_knot,
        spacing=mass.spline_basis_spacing,
    )
    return mass_i, knot_positions


n_draws = 6
scale_knots = 1  # try 0.0 (uniform spacing), 1.0 (mild), 3.0 (strong clustering)
scale_coeffs = 1.5    # try 0.0 (pure power law), 0.5 (mild), 2.0 (very wiggly)

key = jax.random.PRNGKey(0)
keys = jax.random.split(key, n_draws)

m_plot = jnp.logspace(jnp.log10(m_low), jnp.log10(m_high), 500)

realizations = []
for k in keys:
    mass_i, knot_positions = draw_realization(k, mass, scale_knots, scale_coeffs)
    p_m1_vals = p_m1(mass_i, m_plot)
    realizations.append((mass_i, knot_positions, p_m1_vals))

fig, ax = plt.subplots(figsize=(9, 6))

colors = sns.color_palette("husl", n_draws)

for i, (mass_i, knot_positions, p_m1_vals) in enumerate(realizations):
    ax.plot(m_plot, p_m1_vals, color=colors[i], lw=1.8, alpha=0.9,
             label=f'draw {i}')
    # mark this draw's knot positions along a small vertical offset per draw
    y_at_knots = jnp.interp(knot_positions, m_plot, p_m1_vals)
    ax.scatter(knot_positions, y_at_knots, color=colors[i], s=25, zorder=5,
                edgecolor='k', linewidth=0.5)

ax.axvline(m_low, ls='--', c='gray', lw=1, label=r'$m_\mathrm{low}$')
ax.axvline(m_high, ls='--', c='gray', lw=1, label=r'$m_\mathrm{high}$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e-5,1)
ax.set_xlabel(r'$m_1\ [M_\odot]$')
ax.set_ylabel(r'$p(m_1)$')
ax.set_title(f'PLS free-knot draws (scale_knots={scale_knots}, scale_coeffs={scale_coeffs})')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()